In [1]:
# sst import
import xarray as xr
import pandas as pd
from pathlib import Path

# --- NorCal bbox ---
lat_min, lat_max = 38.0, 40.0
lon_min, lon_max = -125.0, -123.0

# --- output ---
out_csv = Path("../../1_DATA/processed/norcal/oisst_norcal_bbox_monthly.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

# --- Monthly OISST via OPeNDAP ---
url = "https://psl.noaa.gov/thredds/dodsC/Datasets/noaa.oisst.v2.highres/sst.mon.mean.nc"
ds = xr.open_dataset(url)  # remote

lat_name = "lat" if "lat" in ds.coords else "latitude"
lon_name = "lon" if "lon" in ds.coords else "longitude"

# dataset lon is usually 0..360
lon = ds[lon_name]
if float(lon.max()) > 180:
    lon_min_use = (lon_min + 360) % 360
    lon_max_use = (lon_max + 360) % 360
else:
    lon_min_use, lon_max_use = lon_min, lon_max

sst = ds["sst"]

# subset bbox (lat might be ascending or descending)
sst_bbox = sst.sel({lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min_use, lon_max_use)})
if sst_bbox.sizes.get(lat_name, 0) == 0:
    sst_bbox = sst.sel({lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min_use, lon_max_use)})

# spatial mean -> monthly time series
sst_m = sst_bbox.mean(dim=[lat_name, lon_name], skipna=True).to_series()
sst_m.index = pd.to_datetime(sst_m.index).tz_localize(None)
sst_m = sst_m.sort_index()
sst_m.name = "sst"

# anomaly vs fixed baseline (1991–2020 if present)
baseline = sst_m.loc["1991-01-01":"2020-12-31"] if len(sst_m.loc["1991":"2020"]) else sst_m
clim = baseline.groupby(baseline.index.month).mean()
sst_anom = sst_m - sst_m.index.month.map(clim)

out = pd.DataFrame({"time": sst_m.index, "sst": sst_m.values, "sst_anom": sst_anom.values})
out.to_csv(out_csv, index=False)

print("Saved:", out_csv.resolve())
print("Range:", out["time"].min(), "to", out["time"].max(), "rows:", len(out))
print(out.head())

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/norcal/oisst_norcal_bbox_monthly.csv
Range: 1981-09-01 00:00:00 to 2026-01-01 00:00:00 rows: 533
        time        sst  sst_anom
0 1981-09-01  13.134558 -0.899032
1 1981-10-01  14.249084  0.528082
2 1981-11-01  13.950231  0.783116
3 1981-12-01  12.970283  0.684740
4 1982-01-01  11.854292  0.110588


In [2]:
# features
import pandas as pd
from pathlib import Path

sst_m_path = Path("../../1_DATA/processed/norcal/oisst_norcal_bbox_monthly.csv")
kelp_path  = Path("../../1_DATA/processed/norcal/kelp_timeseries_norcal_bbox.csv")
out_path   = Path("../../1_DATA/processed/norcal/oisst_features_norcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

# load
sst_m = pd.read_csv(sst_m_path, parse_dates=["time"])
sst_m = sst_m.set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

# quarterly features from monthly
q = sst_m.resample("QS")  # quarter start buckets

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

# lag (1 quarter)
feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

# align to kelp quarters (using q-start)
feat = feat.reindex(kelp_qstart)
feat.index = kelp_times  # set index to kelp timestamps for easy join later

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/norcal/oisst_features_norcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   12.202583        0.542893       0.692359             1.554583
1984-05-15   11.779708       -0.043954       0.570176             0.542893
1984-08-15   13.701208        0.194345       0.285349            -0.043954
1984-11-15   12.798329       -0.259557      -0.129173             0.194345
1985-02-15   10.790242       -0.869447       0.039865            -0.259557
1985-05-15   11.223646       -0.600016       0.043279            -0.869447
1985-08-15   13.544517        0.037654       0.211037            -0.600016
1985-11-15   12.138547       -0.919339      -0.472036             0.037654
1986-02-15   12.403174        0.743484       1.156977            -0.919339
1986-05-15   11.872881        0.049219       0.701599             0.743484


In [3]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_norcalv1_bbox_monthly.csv")
kelp_path  = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/kelp_timeseries_norcalv1_bbox.csv")
out_path   = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_norcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

sst_m = pd.read_csv(sst_m_path, parse_dates=["time"]).set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

q = sst_m.resample("QS")

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

feat = feat.reindex(kelp_qstart)
feat.index = kelp_times

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_norcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   12.176485        0.531792       0.682843                  NaN
1984-05-15   11.759291       -0.046608       0.577767             0.531792
1984-08-15   13.703316        0.221706       0.321739            -0.046608
1984-11-15   12.760182       -0.276030      -0.128920             0.221706
1985-02-15   10.754113       -0.890579       0.021274            -0.276030
1985-05-15   11.178831       -0.627068      -0.006830            -0.890579
1985-08-15   13.476942       -0.004669       0.148303            -0.627068
1985-11-15   12.074263       -0.961948      -0.541627            -0.004669
1986-02-15   12.390114        0.745422       1.148107            -0.961948
1986-05-15   11.843278        0.037379       0.719787             0.745422


In [4]:
# graph rq
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# load monthly NorCal SST (made earlier)
sst_path = Path("../../1_DATA/processed/norcal/oisst_norcal_bbox_monthly.csv")
d = pd.read_csv(sst_path, parse_dates=["time"]).set_index("time").sort_index()

fig_dir = Path("../../5_FIGURES/norcal")
fig_dir.mkdir(parents=True, exist_ok=True)
outpath = fig_dir / "norcal_sst_monthly.png"

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(d.index, d["sst"])
ax.set_title("NorCal OISST SST (monthly mean)")
ax.set_xlabel("Year")
ax.set_ylabel("SST (°C)")
fig.tight_layout()

fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.close(fig)

print("saved:", outpath.resolve(), "bytes:", outpath.stat().st_size)

saved: /Users/tonylin/Documents/kelp_project/5_FIGURES/norcal/norcal_sst_monthly.png bytes: 196220


In [5]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_norcalv1_bbox_monthly.csv")
kelp_path  = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/kelp_timeseries_norcalv1_bbox.csv")
out_path   = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_norcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

sst_m = pd.read_csv(sst_m_path, parse_dates=["time"]).set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

q = sst_m.resample("QS")

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

feat = feat.reindex(kelp_qstart)
feat.index = kelp_times

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_norcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   12.176485        0.531792       0.682843                  NaN
1984-05-15   11.759291       -0.046608       0.577767             0.531792
1984-08-15   13.703316        0.221706       0.321739            -0.046608
1984-11-15   12.760182       -0.276030      -0.128920             0.221706
1985-02-15   10.754113       -0.890579       0.021274            -0.276030
1985-05-15   11.178831       -0.627068      -0.006830            -0.890579
1985-08-15   13.476942       -0.004669       0.148303            -0.627068
1985-11-15   12.074263       -0.961948      -0.541627            -0.004669
1986-02-15   12.390114        0.745422       1.148107            -0.961948
1986-05-15   11.843278        0.037379       0.719787             0.745422


In [5]:
# graph rq
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# load monthly NorCal SST (made earlier)
sst_path = Path("../../1_DATA/processed/oisst_norcalv1_bbox_monthly.csv")
d = pd.read_csv(sst_path, parse_dates=["time"]).set_index("time").sort_index()

fig_dir = Path("../../5_FIGURES")
fig_dir.mkdir(parents=True, exist_ok=True)
outpath = fig_dir / "norcal_sst_monthly.png"

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(d.index, d["sst"])
ax.set_title("NorCal OISST SST (monthly mean)")
ax.set_xlabel("Year")
ax.set_ylabel("SST (°C)")
fig.tight_layout()

fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.close(fig)

print("saved:", outpath.resolve(), "bytes:", outpath.stat().st_size)

saved: /Users/tonylin/Documents/kelp_project/5_FIGURES/norcal_sst_monthly.png bytes: 192165


In [6]:
# graph all
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

RAW_DIR = Path("../../1_DATA/raw")
files = sorted([p for p in RAW_DIR.glob("oisst_norcal_*.nc") if p.stat().st_size > 0])

print("Found files:", len(files))
print("First/last:", files[0].name, files[-1].name)

def norm_ds(ds: xr.Dataset) -> xr.Dataset:
    # normalize names
    if "latitude" in ds.dims:  ds = ds.rename({"latitude": "lat"})
    if "longitude" in ds.dims: ds = ds.rename({"longitude": "lon"})
    if "latitude" in ds.coords and "lat" not in ds.coords:  ds = ds.rename({"latitude": "lat"})
    if "longitude" in ds.coords and "lon" not in ds.coords: ds = ds.rename({"longitude": "lon"})
    if "zlev" in ds.dims:
        ds = ds.isel(zlev=0, drop=True)
    return ds

rows = []
bad = 0

for i, f in enumerate(files, 1):
    try:
        ds = xr.open_dataset(f)
        ds = norm_ds(ds)
        if "sst" not in ds:
            ds.close()
            bad += 1
            continue

        s = ds["sst"].mean(dim=[d for d in ("lat", "lon") if d in ds["sst"].dims], skipna=True)
        ser = s.to_pandas()
        ser.name = "sst"
        rows.append(ser)

        ds.close()
    except Exception as e:
        bad += 1
        print("Skip (error):", f.name, "->", repr(e))

    if i % 50 == 0 or i == len(files):
        print(f"Read {i}/{len(files)}")

sst = pd.concat(rows).sort_index()
sst = sst[~sst.index.duplicated(keep="first")]  # drop overlaps at chunk boundaries

print("Bad files skipped:", bad)
print("Points:", len(sst))
print("Range:", sst.index.min(), "->", sst.index.max())

# Plot (all points)
fig_dir = Path("../../5_FIGURES")
fig_dir.mkdir(parents=True, exist_ok=True)
outpath = fig_dir / "norcal_sst_all_points.png"

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(sst.index, sst.values)
ax.set_title("NorCal OISST SST (all downloaded timesteps, spatial mean)")
ax.set_xlabel("Year")
ax.set_ylabel("SST (°C)")
fig.tight_layout()
fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.close(fig)

print("saved:", outpath.resolve(), "bytes:", outpath.stat().st_size)

Found files: 508
First/last: oisst_norcal_1984-01-01_1984-01-28.nc oisst_norcal_2025-12-01_2025-12-31.nc
Read 50/508
Read 100/508
Read 150/508
Read 200/508
Read 250/508
Read 300/508
Read 350/508
Read 400/508
Read 450/508
Read 500/508
Read 508/508
Bad files skipped: 0
Points: 15257
Range: 1984-01-01 12:00:00 -> 2025-12-31 12:00:00
saved: /Users/tonylin/Documents/kelp_project/5_FIGURES/norcal_sst_all_points.png bytes: 179384
